# 04 — U-Net on PS-OCT, trained on MRI + classical consensus labels

There are no hand labels, so the labels come from the segmentations we have. For every slide-deck section, the warped
MRI labels (`02_mri_pseudo_labels.ipynb`) and the classical PS-OCT segmentation (`otsu`, `brainseg segment`) are
combined: pixels where they agree are kept, the rest are ignored (`brainseg.mri_labels.consensus`). A 2D U-Net
(`brainseg.methods.unet`) sees only PS-OCT retardance and reflectivity.

It is trained on one brain and tested on the other, in both directions, so the test brain's labels are never seen.
It is scored like the classical methods in `03_benchmark_mri_labels.ipynb`, on the same 20 sections per brain, against
the MRI labels alone. **Scores are agreement with the MRI, not accuracy**, and the U-Net learned partly from the
otsu output, so agreement with otsu is not evidence either. Requires the `unet` extra (PyTorch) and notebook 03's
outputs.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

from brainseg.benchmark import evaluate
from brainseg.compare import compare
from brainseg.methods import unet
from brainseg.methods.classical import segment
from brainseg.mri_labels import consensus

DATA, OUT = Path("../data"), Path("../outputs")
BENCH, CACHE, MODELS = DATA / "benchmark_mri", DATA / "unet", OUT / "unet"
SUBJECTS = {"vlad": 0, "moe": 1}  # slide-deck axis that indexes sections
OTHER = {"vlad": "moe", "moe": "vlad"}
SETTINGS = {"iterations": 4000, "patch": 256, "batch": 16, "lr": 1e-3, "width": 16, "depth": 4, "seed": 0}
CACHE.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
print("device:", unet.device())

## Training labels

Every section with MRI-labelled tissue. `x` holds retardance and reflectivity; `otsu` the classical labels;
`labels` the consensus. Cached in `data/unet/` (about 5 GB).

In [ ]:
def load_brain(s):
    paths = {k: CACHE / f"{s}_{k}.npy" for k in ("x", "otsu", "mri", "labels", "index")}
    if not all(p.exists() for p in paths.values()):
        axis = SUBJECTS[s]
        mri = np.moveaxis(np.asarray(nib.load(OUT / f"{s}_labels_psoct.nii.gz").dataobj), axis, 0)
        x = np.stack([np.moveaxis(np.asarray(nib.load(DATA / s / "warp" / f"{c}_slide_deck.nii.gz").dataobj, dtype=np.float32), axis, 0)
                      for c in ("Ret", "Ref")], -1)
        index = np.flatnonzero(np.isin(mri, (1, 2, 3)).any((1, 2)))
        otsu = np.stack([segment(x[i], "otsu", retardance=0, reflectivity=1, pixel_size=0.055) for i in index])
        for k, v in {"x": x[index], "otsu": otsu, "mri": mri[index], "labels": consensus(mri[index], otsu), "index": index}.items():
            np.save(paths[k], v)
    return {k: np.load(p, mmap_mode="r") for k, p in paths.items()}

brains = {s: load_brain(s) for s in SUBJECTS}
for s, b in brains.items():
    mri, labels = np.asarray(b["mri"]), np.asarray(b["labels"])
    kept = {name: (labels == c).sum() / max((mri == c).sum(), 1) for c, name in enumerate(("background", "gray", "white", "fluid"))}
    print(f"{s}: {len(b['index'])} sections; consensus keeps " + ", ".join(f"{k} {v:.0%}" for k, v in kept.items()) + " of MRI labels")

## Train on one brain, test on the other

Models are cached in `outputs/unet/`; delete one to retrain. About 15-20 minutes per direction on an M-series GPU.
Training on MPS is not bit-for-bit reproducible.

In [ ]:
models = {}
fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
for train in SUBJECTS:
    path = MODELS / f"{train}_model.pt"
    if not path.exists():
        b = brains[train]
        model, losses = unet.fit([np.asarray(a) for a in b["x"]], [np.asarray(y) for y in b["labels"]], **SETTINGS)
        unet.save(model, path, train_sample_ids=[train], labels="consensus of MRI and otsu", losses=losses, **SETTINGS)
    models[train], meta = unet.load(path)
    ax.plot(np.convolve(meta["losses"], np.ones(50) / 50, "valid"), label=f"trained on {train}")
ax.set(xlabel="iteration", ylabel="cross-entropy (50-step mean)", ylim=(0, None))
ax.legend()

## Score on the benchmark sections

The test brain's rows from notebook 03's manifests, with all MRI labels and with the boundary band ignored. Run names
say which brain the model was trained on.

`unet_masked` keeps the classical tissue mask: it starts from the otsu labels and only where both otsu and the U-Net
say gray or white matter does it take the U-Net's choice. Background, tissue edge and fluid are otsu's, so any
difference from otsu is in the gray/white split alone.

In [ ]:
CLASSICAL = {"retardance": 0, "reflectivity": 1, "pixel_size": 0.055}


def masked(unet_labels, otsu_labels):
    """The U-Net's gray/white choice inside the otsu gray and white matter; otsu everywhere else."""
    out = otsu_labels.copy()
    split = np.isin(otsu_labels, (1, 2)) & np.isin(unet_labels, (1, 2))
    out[split] = unet_labels[split]
    return out


runs = []
for train, model in models.items():
    test = OTHER[train]
    for suffix in ("", "_band"):
        rows = [r for r in csv.DictReader(open(BENCH / f"manifest{suffix}.csv")) if r["sample_id"] == test]
        manifest = BENCH / f"manifest{suffix}_{test}.csv"
        with manifest.open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        predictors = {
            "unet": lambda a, m=model: unet.predict(m, a),
            "unet_masked": lambda a, m=model: masked(unet.predict(m, a), segment(a, "otsu", **CLASSICAL)),
        }
        for name, predict in predictors.items():
            run = OUT / "mri_benchmark" / f"{name}_from_{train}{suffix}"
            evaluate(manifest, "test", run, name, predict, train_sample_ids=[train])
            runs.append(run)

classical = [OUT / "mri_benchmark" / f"{m}{suffix}" for suffix in ("", "_band") for m in ("otsu", "gmm", "kmeans")]
table = compare(classical + runs, OUT / "mri_benchmark" / "table_unet.csv")
for r in table:
    if r["metric"] == "dice" and r["class"] in ("background", "gray_matter", "white_matter"):
        print(f"{r['run']:27s} {r['sample_id']:5s} {r['class']:13s} Dice {r['mean']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]")

In [ ]:
classes = ("background", "gray_matter", "white_matter")
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained", sharey=True)
for (suffix, variant), row in zip((("", "all labels"), ("_band", "boundary band ignored")), axes):
    for ax, s in zip(row, SUBJECTS):
        names = [f"otsu{suffix}", f"gmm{suffix}", f"unet_from_{OTHER[s]}{suffix}", f"unet_masked_from_{OTHER[s]}{suffix}"]
        for k, name in enumerate(names):
            sel = {r["class"]: r for r in table if r["run"] == name and r["sample_id"] == s and r["metric"] == "dice"}
            mean = [sel[c]["mean"] for c in classes]
            err = [[m - sel[c]["ci_low"] for m, c in zip(mean, classes)], [sel[c]["ci_high"] - m for m, c in zip(mean, classes)]]
            ax.bar(np.arange(3) + (k - 1.5) * 0.2, mean, 0.2, yerr=err, capsize=2, label=name.removesuffix(suffix))
        ax.set(xticks=range(3), xticklabels=classes, title=f"{s}, {variant}: Dice vs MRI (95% CI)", ylim=(0, 1))
axes[0, 0].legend()

## Checks without ground truth

Predict every section of each held-out brain, then compare U-Net, masked U-Net, otsu and the MRI labels on:

- **Section-to-section consistency:** Dice of gray and of white matter between neighbouring sections. Tissue
  changes little over one section, so lower values mean more noise. The MRI labels are smooth by construction.
- **Whole-brain composition:** gray, white and fluid as fractions of labelled tissue.
- **Agreement between methods:** pixel agreement inside the tissue, to see how far the U-Net departs from otsu.

In [ ]:
def dice(a, b):
    total = a.sum() + b.sum()
    return 2 * (a & b).sum() / total if total else np.nan

predictions = {}
for train, model in models.items():
    test = OTHER[train]
    predictions[test] = np.stack([unet.predict(model, np.asarray(a)) for a in brains[test]["x"]])
    np.save(CACHE / f"{test}_unet.npy", predictions[test])

for s in SUBJECTS:
    b = brains[s]
    otsu = np.asarray(b["otsu"])
    sets = {"unet": predictions[s], "unet_masked": masked(predictions[s], otsu), "otsu": otsu, "mri": np.asarray(b["mri"])}
    print(s)
    for name, labels in sets.items():
        neighbour = {c: np.nanmean([dice(labels[i] == c, labels[i + 1] == c) for i in range(len(labels) - 1)]) for c in (1, 2)}
        tissue = np.isin(labels, (1, 2, 3))
        fractions = [(labels == c).sum() / tissue.sum() for c in (1, 2, 3)]
        print(f"  {name:11s} neighbour Dice gray {neighbour[1]:.3f} white {neighbour[2]:.3f};"
              f" gray/white/fluid {fractions[0]:.0%}/{fractions[1]:.0%}/{fractions[2]:.1%}")
    both = np.isin(sets["unet"], (1, 2, 3)) & np.isin(sets["otsu"], (1, 2, 3))
    print(f"  unet vs otsu: same label on {(sets['unet'] == sets['otsu'])[both].mean():.0%} of pixels both call tissue")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), layout="constrained")
for row, s in zip(axes, SUBJECTS):
    b = brains[s]
    for ax, i in zip(row, np.linspace(0.25, 0.75, 3) * len(b["index"])):
        i = int(i)
        ret = np.asarray(b["x"][i, ..., 0])
        ax.imshow(np.rot90(ret), cmap="gray", vmin=0, vmax=np.percentile(ret[ret > 0], 99))
        for labels, colour in ((b["mri"], "tab:orange"), (b["otsu"], "tab:green"), (predictions[s], "tab:cyan")):
            ax.contour(np.rot90(np.asarray(labels[i]) == 2), levels=[0.5], colors=colour, linewidths=0.6)
        ax.set(xticks=[], yticks=[], title=f"{s} deck {b['index'][i]}: white, MRI / otsu / U-Net")

## Findings

One training run per direction (4000 steps, seed 0). Scores are agreement with the MRI labels, not accuracy.

- **Trained on Moe, tested on Vlad, the U-Net matches the classical methods:** gray Dice 0.81 and white 0.75
  (otsu 0.79 / 0.74), or 0.84 / 0.80 with the boundary band ignored (otsu 0.83 / 0.79). The intervals overlap, so
  it is not measurably better.
- **Trained on Vlad, tested on Moe, it is worse on its own:** background 0.76, gray 0.78, white 0.75 (otsu 0.90 /
  0.82 / 0.78). It labels Moe's bright gel, which shows retardance artifacts and a tile pattern, as tissue, mostly
  white matter (overlays above). Vlad's gel is dark and uniform, so a Vlad-trained model never saw it. The two
  brains were processed with different pipeline versions and cut in different planes (`docs/data_notes.md`).
- **The classical tissue mask removes that failure.** With otsu's tissue edge and the U-Net's gray/white split
  (`unet_masked`), Moe scores background 0.90, gray 0.82, white 0.77, the same as otsu, and its section-to-section
  consistency recovers (neighbour Dice 0.90 gray, 0.86 white vs otsu 0.90 / 0.87). On Vlad the mask changes
  nothing measurable. So the U-Net's error was entirely at the tissue edge, and its gray/white split is
  indistinguishable from otsu's.
- **Composition is close to the MRI's** for every method (gray/white 63-69% / 31-35% of tissue; MRI 64/35-36%). The
  U-Net alone predicts **no fluid**, because the consensus labels contain almost none; the masked variant takes
  otsu's fluid.
- **Inside tissue the U-Net mostly reproduces otsu** (same label on 94-95% of pixels both call tissue), as expected
  from training on labels where otsu and the MRI agree. A model trained on these labels cannot show a gain over
  them on this benchmark.